In [1]:
import re
import urllib.request
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

In [2]:
BASE_URL = "ftp://ftp.ebi.ac.uk/biostudies/fire/E-MTAB-/061/E-MTAB-5061/Files/"
COUNT_FILE = "pancreas_refseq_rpkms_counts_3514sc.txt"
SDRF_FILE = "E-MTAB-5061.sdrf.txt"

CACHE_DIR = Path("raw_data")

### Helper functions

- fetch data
- Get count data
- Read the metadata to extract the need data

In [51]:
#Helper functions

def fetch_data(local=None, cache=CACHE_DIR):
    if local:
        return Path(local)
    cache.mkdir(parents=True, exist_ok=True)
    path = cache / name
    if not path.exists():
        print(f"Downloading {name} ...")
        urllib.request.urlretrieve(BASE_URL + name, path)
    return path

def read_counts(path):
    with open(path) as f:
        cells = [c.strip() for c in f.readline().rstrip("\n").split("\t")[1:]]  # first field is a label
    n = len(cells)
    count_cols = list(range(2 + n, 2 + 2 * n))  # skip symbol, RefSeq IDs and the RPKM block
    df = pd.read_csv(
        path, sep="\t", header=None, skiprows=1, usecols=[0] + count_cols, index_col=0,
        dtype={c: np.int32 for c in count_cols},
    )
    df.columns = cells
    df = df[~df.index.astype(str).str.startswith("ERCC")]  # drop spike-ins
    df = df.groupby(level=0).sum()  # gene symbols are not unique; sum duplicates
    df.index.name = None
    df.columns.name = None
    print(f"Counts: {df.shape[0]} genes x {df.shape[1]} cells")
    return df


def read_metadata(path, cells):
    sdrf = pd.read_csv(path, sep="\t")
    sdrf.columns = [str(c).strip() for c in sdrf.columns]
    cells = pd.Index(cells)

    # Find the SDRF column whose values are the cell names used in the count file
    best, best_n = None, 0

    for col in sdrf.columns:
        n = int(
            sdrf[col]
            .astype(str)
            .str.strip()
            .isin(cells)
            .sum()
        )

        if n > best_n:
            best, best_n = col, n

    if best is None:
        raise ValueError(
            "No SDRF column matches the cell names.\n"
            f"Example cell names: {list(cells[:3])}\n"
            f"SDRF columns: {list(sdrf.columns)}"
        )

    print(f"Matching cells on SDRF column '{best}' ({best_n} rows)")
    
    sdrf[best] = sdrf[best].astype(str).str.strip()
    sdrf = sdrf.drop_duplicates(best).set_index(best)

    # Extract the metadata we need
    obs = pd.DataFrame(index=cells)
    

    obs["cell type"] = (
        sdrf["Characteristics [inferred cell type]"]
        .reindex(cells)
        .values
    )

    obs["individual"] = (
        sdrf["Characteristics [individual]"]
        .reindex(cells)
        .values
    )

    obs["disease"] = (
        sdrf["Characteristics [disease]"]
        .reindex(cells)
        .values
    )

    obs["quality"] = (
        sdrf["Characteristics [submitted single cell quality]"]
        .reindex(cells)
        .values
    )

    missing = int(obs["cell type"].isna().sum())

    print(f"Metadata matched for {len(obs) - missing} of {len(obs)} cells")

    print(f"SDRF columns: {list(sdrf.columns)}")

    return obs


### Fetch data 

In [24]:
counts_path = fetch_data(COUNT_FILE)
sdrf_path = fetch_data(SDRF_FILE)

print(counts_path)
print(sdrf_path)

pancreas_refseq_rpkms_counts_3514sc.txt
E-MTAB-5061.sdrf.txt


In [25]:
df = read_counts(f"raw_data/{counts_path}")

Counts: 25444 genes x 3514 cells


In [18]:
df.head()

,HP1502401_N13,HP1502401_D14,HP1502401_F14,HP1502401_J13,HP1502401_B13,HP1502401_H13,HP1502401_J14,HP1502401_B14,HP1502401_A14,HP1502401_C14,...,HP1525301T2D_O10,HP1526901T2D_H2,HP1526901T2D_I16,HP1526901T2D_F7,HP1526901T2D_I23,HP1525301T2D_K3,HP1525301T2D_J10,HP1526901T2D_N8,HP1526901T2D_O11,HP1526901T2D_A8
A1BG,0,0,0,0,0,13,36,19,0,43,...,0,0,10,8,0,6,2,6,1,2
A1BG-AS1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
A1CF,0,0,0,0,0,0,0,26,0,427,...,0,0,17,5,0,26,13,28,0,70
A2M,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
A2M-AS1,0,0,0,0,0,0,0,0,0,0,...,0,2,0,0,0,0,0,0,0,0


In [19]:
df.shape

(25444, 3514)

In [52]:
obs = read_metadata(f"raw_data/{sdrf_path}", df.columns)

Matching cells on SDRF column 'Source Name' (3514 rows)
Metadata matched for 3514 of 3514 cells
SDRF columns: ['Characteristics [organism]', 'Characteristics [individual]', 'Characteristics [sex]', 'Characteristics [age]', 'Unit [time unit]', 'Characteristics [body mass index]', 'Characteristics [organism status]', 'Characteristics [clinical information]', 'Characteristics [organism part]', 'Characteristics [cell type]', 'Characteristics [disease]', 'Characteristics [biosource provider]', 'Characteristics [single cell quality]', 'Characteristics [submitted single cell quality]', 'Characteristics [inferred cell type]', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Material Type', 'Protocol REF', 'Protocol REF.1', 'Performer', 'Extract Name', 'Comment[single cell isolation]', 'Comment[input molecule]', 'Comment[library construction]', 'Comment[end bias]', 'Comment[primer]', 'Comment[LIBRARY_STRAND]', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRATEGY]', 'C

In [53]:
obs.head()

,cell type,individual,disease,quality
HP1502401_N13,not applicable,H2,normal,low quality cell
HP1502401_D14,not applicable,H2,normal,low quality cell
HP1502401_F14,not applicable,H2,normal,low quality cell
HP1502401_J13,not applicable,H2,normal,low quality cell
HP1502401_B13,not applicable,H2,normal,low quality cell


### Create the AnnData object and save it

In [58]:
adata = ad.AnnData(
    X=sparse.csr_matrix(df.T.values.astype(np.float32)),
    obs=obs,
    var=pd.DataFrame(index=df.index.astype(str)),
)

OUT_FILE = Path("segerstolpe.h5ad")

adata.write_h5ad(OUT_FILE)

In [55]:
adata.obs["cell type"].value_counts(dropna=False)

cell type
not applicable                 1305
alpha cell                      886
ductal cell                     386
beta cell                       270
gamma cell                      197
acinar cell                     185
delta cell                      114
PSC cell                         54
unclassified endocrine cell      41
co-expression cell               39
endothelial cell                 16
epsilon cell                      7
mast cell                         7
MHC class II cell                 5
unclassified cell                 2
Name: count, dtype: int64

In [57]:
adata.obs["quality"].value_counts(dropna=False)

quality
OK                      2209
low quality cell        1177
control, empty well       96
control, 2-cell well      32
Name: count, dtype: int64

In [56]:
pd.crosstab(
    adata.obs["quality"],
    adata.obs["cell type"],
    dropna=False
)

cell type,MHC class II cell,PSC cell,acinar cell,alpha cell,beta cell,co-expression cell,delta cell,ductal cell,endothelial cell,epsilon cell,gamma cell,mast cell,not applicable,unclassified cell,unclassified endocrine cell
quality,,,,,,,,,,,,,,,
OK,5,54,185,886,270,39,114,386,16,7,197,7,0,2,41
"control, 2-cell well",0,0,0,0,0,0,0,0,0,0,0,0,32,0,0
"control, empty well",0,0,0,0,0,0,0,0,0,0,0,0,96,0,0
low quality cell,0,0,0,0,0,0,0,0,0,0,0,0,1177,0,0
